In [1]:
from itertools import product
import numpy as np

!pip install qiskit
from qiskit.quantum_info import Operator
from qiskit.circuit import library


def make_square(matrix):
    # Pad with 0 to make square matrix
    if matrix.shape[0] != matrix.shape[1]:
        if matrix.shape[0] > matrix.shape[1]:
            # Pad columns
            pad_width = [(0, 0), (0, matrix.shape[0] - matrix.shape[1])]
        else:
            # Pad rows
            pad_width = [(0, matrix.shape[1] - matrix.shape[0]), (0, 0)]
        matrix = np.pad(matrix, pad_width)
    return matrix

def get_pauli_bases(dimension):
    pauli = {
        'x': Operator(library.XGate().to_matrix()),
        'y': Operator(library.YGate().to_matrix()),
        'z': Operator(library.ZGate().to_matrix()),
        'i': Operator(library.IGate().to_matrix())
    }

    bases = {}
    # Creates bases matrix in dimension mentioned in parameter
    # For dimension 1, bases = {I, X, Y, Z}
    # For dimension 2, bases = {II, IX, IY, IZ, XX, XY, XZ, YY, YZ, ZZ}
    if dimension == 1:
        return pauli
    else:
        for permutation in product(*[list(pauli.keys())]*dimension):
            permutation = "".join(permutation)
            bases[permutation] = pauli[permutation[0]]
            for idx in range(1, len(permutation)):
                bases[permutation] = bases[permutation].tensor(pauli[permutation[idx]])
        return bases

def linear_combination_pauli(matrix):
    matrix = make_square(matrix)
    matrix_len = matrix.shape[0]

    # Assuming matrix dimension is in power of 2
    nqubits = int(np.log2(matrix_len))

    bases = get_pauli_bases(nqubits)
    decomposition = {}
    for base, base_matrix in bases.items():
        decomposition[base] = np.trace(np.dot(base_matrix, matrix)) / matrix_len
    return decomposition, bases

def validate_decomposition(decomposition, bases, original):
    created = sum(coeff*matrix.data for coeff, matrix in zip(decomposition.values(), bases.values()))
    created = np.around(created, 7)
    print(created.shape)
    print(created)
    # return (created == original).all()
    return np.allclose(original, created, atol=1e-07)





A = np.array([[1, 0, 0, 0], [1, 1, 1, 1], [0, 0, 2, 2], [0, 0, 2, 4]])
# A = np.random.rand(16, 16)
# A = np.loadtxt("/content/solution_fourier.txt", delimiter=',')
print("A MATRIX")
print(A.shape)
print(np.linalg.cond(A))
print(np.linalg.norm(A))
print(np.allclose(A, A.T, atol=0.1))
print(np.linalg.matrix_rank(A))
A=A/np.linalg.norm(A)

# b = np.loadtxt('b_vector.txt', delimiter=',')
# b = np.array([1, 2, 3, 4])
# print("B VECTOR")
# print(b.shape)
# print(b)

# x = np.loadtxt('solution_matrix.txt', delimiter=',')
# print("X VECTOR")
# print(x.shape)
# print(x)




decomp, bases = linear_combination_pauli(A)
np.set_printoptions(threshold=np.inf, precision = 10)
print(validate_decomposition(decomp, bases, A))
print("Number of unitaries used", sum(1 for value in decomp.values() if value != 0j))
print(decomp)
print(bases)
print(len(bases))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 4.7 MB/s eta 0:00:00
A MATRIX
(4, 4)
9.392046612362055
5.744562646538029
False
4
(4, 4)
[[ 0.1740777+0.j -0.       +0.j  0.       +0.j  0.       +0.j]
 [ 0.1740777+0.j  0.1740777+0.j  0.1740777+0.j  0.1740777+0.j]
 [ 0.       +0.j  0.       +0.j  0.3481553+0.j  0.3481553+0.j]
 [ 0.       +0.j  0.       +0.j  0.3481553+0.j  0.6963106+0.j]]
True
Number of unitaries used 16
{'xx': np.complex128(0.04351941398892446+0j), 'xy': np.complex128(-0.04351941398892446j), 'xz': np.complex128(-0.04351941398892446+0j), 'xi': np.complex128(0.04351941398892446+0j), 'yx': np.complex128(0.04351941398892446j), 'yy': np.complex128(0.04351941398892446+0j), 'yz': np.complex128(-0.04351941398892446j), 'yi': np.complex128(0.04351941398892446j), 'zx': np.complex128(-0.1305582419667734+0j), 'zy': np.compl